<a href="https://colab.research.google.com/github/op1154/integration/blob/main/Strawberry_MCP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install fastapi uvicorn strawberry-graphql httpx mcp


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 358.1/358.1 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 368.7/368.7 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.9/347.9 kB 18.1 MB/s eta 0:00:00


In [3]:
from typing import List, Optional
import strawberry
from fastapi import FastAPI, APIRouter

# Banco de dados simulado na memória
DB_ALUNOS = [
    {"id": "1", "nome": "Ana Silva", "idade": 21},
    {"id": "2", "nome": "Bruno Costa", "idade": 23}
]

# Tipos GraphQL
@strawberry.type
class Aluno:
    id: str
    nome: str
    idade: int


In [4]:
@strawberry.type
class Query:
    @strawberry.field
    def aluno(self, id: str) -> Optional[Aluno]:
        for a in DB_ALUNOS:
            if a["id"] == id:
                return Aluno(**a)
        return None

    @strawberry.field
    def alunos(self) -> List[Aluno]:
        return [Aluno(**a) for a in DB_ALUNOS]

    # Requisito: Adicione o campo totalAlunos ao tipo Query
    @strawberry.field
    def total_alunos(self) -> int:
        return len(DB_ALUNOS)

@strawberry.type
class Mutation:
    # Requisito: Crie a mutation criarAluno
    @strawberry.mutation
    def criar_aluno(self, nome: str, idade: int) -> Aluno:
        novo_id = str(len(DB_ALUNOS) + 1)
        novo_aluno = {"id": nuevo_id, "nome": nome, "idade": idade}
        DB_ALUNOS.append(novo_aluno)
        return Aluno(**novo_aluno)

    @strawberry.mutation
    def atualizar_aluno(self, id: str, nome: Optional[str] = None, idade: Optional[int] = None) -> Optional[Aluno]:
        for a in DB_ALUNOS:
            if a["id"] == id:
                if nome: a["nome"] = nome
                if idade: a["idade"] = idade
                return Aluno(**a)
        return None

    @strawberry.mutation
    def remover_aluno(self, id: str) -> bool:
        global DB_ALUNOS
        inicial = len(DB_ALUNOS)
        DB_ALUNOS = [a for a in DB_ALUNOS if a["id"] != id]
        return len(DB_ALUNOS) < inicial

schema = strawberry.Schema(query=Query, mutation=Mutation)


In [5]:
from strawberry.fastapi import GraphQLRouter
from pydantic import BaseModel

app = FastAPI()

# Acopla o painel do GraphiQL e a rota GraphQL
graphql_app = GraphQLRouter(schema)
app.include_router(graphql_app, prefix="/graphql")

# Equivalente REST com POST /api/alunos
class AlunoInputREST(BaseModel):
    nome: str
    idade: int

@app.post("/api/alunos")
def criar_aluno_rest(aluno: AlunoInputREST):
    novo_id = str(len(DB_ALUNOS) + 1)
    novo_aluno = {"id": novo_id, "nome": aluno.nome, "idade": aluno.idade}
    DB_ALUNOS.append(novo_aluno)
    return novo_aluno


In [6]:
import threading
import uvicorn
import time

def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

# Inicia o servidor em uma thread separada
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# Aguarda 2 segundos para o servidor subir completamente
time.sleep(2)
print("Servidor rodando localmente no Colab em http://127.0.0.1:8000")


Servidor rodando localmente no Colab em http://127.0.0.1:8000


In [7]:
import httpx

query_aliases = """
query {
  primeiro: aluno(id: "1") {
    nome
  }
  todos: alunos {
    id
    nome
  }
  total: totalAlunos
}
"""

res = httpx.post("http://127.0.0.1:8000", json={"query": query_aliases})
print("--- Teste GraphQL (Aliases e totalAlunos) ---")
print(res.json())


--- Teste GraphQL (Aliases e totalAlunos) ---
{'detail': 'Not Found'}


TESTE RESTFUL

In [8]:
res_rest = httpx.post("http://127.0.0.1:8000/api/alunos", json={"nome": "Carlos Sousa", "idade": 25})
print("\n--- Teste Endpoint REST POST ---")
print(res_rest.json())


--- Teste Endpoint REST POST ---
{'id': '3', 'nome': 'Carlos Sousa', 'idade': 25}


# Servidor MCP

In [16]:
# Instala a versão estável compatível do MCP (versão 1.x que possui o FastMCP)
!pip install "mcp[cli]<2" --upgrade -q

# Força a limpeza de módulos antigos que ficaram presos na memória do Kernel do Jupyter
import sys
for modulo in list(sys.modules.keys()):
    if modulo.startswith("mcp"):
        del sys.modules[modulo]

from mcp.server.fastmcp import FastMCP

# Cria o servidor MCP
mcp_server = FastMCP("Gerenciador de Alunos")

# Expõe a funcionalidade criarAluno como Tool
@mcp_server.tool()
def criar_aluno_tool(nome: str, idade: int) -> str:
    """Adiciona um novo aluno ao banco de dados através da lógica definida."""
    novo_id = str(len(DB_ALUNOS) + 1)
    novo_aluno = {"id": novo_id, "nome": nome, "idade": idade}
    DB_ALUNOS.append(novo_aluno)
    return f"Aluno {nome} criado com sucesso via MCP! ID: {novo_id}"

# Cliente Python chamando a ferramenta de forma assíncrona utilizando 'await'
print("--- Chamando a Tool pelo Cliente Python MCP ---")
resultado_mcp = await mcp_server.call_tool("criar_aluno_tool", arguments={"nome": "Mariana Lima", "idade": 22})
print(resultado_mcp)

--- Chamando a Tool pelo Cliente Python MCP ---
([TextContent(type='text', text='Aluno Mariana Lima criado com sucesso via MCP! ID: 4', annotations=None, meta=None)], {'result': 'Aluno Mariana Lima criado com sucesso via MCP! ID: 4'})


In [18]:
# 1. Instalar o pacote oficial do ngrok para Python
!pip install pyngrok -q
from google.colab import userdata

# 2. Configurar a sua Authtoken do ngrok
try:
    # Tenta obter dos segredos do Colab
    NGROK_TOKEN = userdata.get('NGROK_TOKEN')
except Exception:
    # Se não encontrar, coloque seu token real aqui como string, por exemplo: NGROK_TOKEN = "seu-token-aqui"
    NGROK_TOKEN = None

if not NGROK_TOKEN:
    print("❌ ERRO: NGROK_TOKEN não foi encontrado nos segredos do Colab.")
    print("Como resolver:")
    print("1. Vá no menu lateral esquerdo do Colab e clique no ícone de chave (Secrets).")
    print("2. Adicione um segredo com o nome NGROK_TOKEN e o valor do seu token do ngrok.")
    print("3. Ative a permissão de acesso para o notebook e execute novamente esta célula.")
    print("Ou configure a variável NGROK_TOKEN diretamente nesta célula como uma string.")
else:
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_TOKEN)

    # 3. Abrir o túnel HTTP apontando para a porta do seu servidor FastAPI (8000)
    try:
        # Fecha conexões antigas caso a célula seja reexecutada
        ngrok.disconnect(public_url)
    except NameError:
        pass

    # Cria o novo túnel
    public_url = ngrok.connect(8000).public_url

    print("🚀 Seu servidor já está acessível publicamente!")
    print(f"🔗 Link da API REST e MCP: {public_url}")
    print(f"✨ Link para abrir o painel GraphiQL: {public_url}/graphql")

🚀 Seu servidor já está acessível publicamente!
🔗 Link da API REST e MCP: https://abnormal-spider-handiwork.ngrok-free.dev
✨ Link para abrir o painel GraphiQL: https://abnormal-spider-handiwork.ngrok-free.dev/graphql
